In [181]:
# import libraries. 
from langchain_core.tools import InjectedToolArg
from typing import Annotated
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_cohere import ChatCohere
import requests
import os
import json

In [182]:
# import the api key for using the model. 
os.environ['COHERE_API_KEY'] = "ADD YOUR OWN KEY"

### Creating the tool.

In [183]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool 
def get_conversion_factor(base_currency : str, target_currency : str) -> float: 
    """
        This function is responsible for finding the conversion factor between the base currency and target currency. 
    """

    url = f'https://v6.exchangerate-api.com/v6/ADD_YOUR_KEY_HERE/pair/{base_currency}/{target_currency}'

    response = requests.get(url)
    
    # print(response.json())
    return response.json()

@tool 
def currency_conversion(base_currency_value : int, conversion_rate : Annotated[float, InjectedToolArg]) -> float: 
    """
        This function is responsible for converting the base currency value to the target currency value, using the current currency rate.
    """
    
    return base_currency_value * conversion_rate

In [184]:
get_conversion_factor.invoke({'base_currency' : 'USD', 'target_currency' : 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1787011201,
 'time_last_update_utc': 'Tue, 18 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1787097601,
 'time_next_update_utc': 'Wed, 19 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.695}

In [185]:
currency_conversion.invoke({'base_currency_value' : 10, 'conversion_rate' : 95.695})

956.9499999999999

### Tool Binding.

In [186]:
# create the llm. 
chat_model = ChatCohere(
    model="command-a-03-2025",
    temperature=0
)

In [187]:
model_with_tools = chat_model.bind_tools([get_conversion_factor, currency_conversion])

In [188]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 USD to INR')]

In [189]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [190]:
ai_message = model_with_tools.invoke(messages)

In [191]:
ai_message

AIMessage(content='I will first find the conversion factor between INR and USD. Then, I will use this factor to convert 10 USD to INR.', additional_kwargs={'id': '15b9ca31-f499-4bf7-8b25-19113ec151d2', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will first find the conversion factor between INR and USD. Then, I will use this factor to convert 10 USD to INR.', 'tool_calls': [{'id': 'get_conversion_factor_xzezjyvqevv2', 'type': 'function', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency":"INR","target_currency":"USD"}'}}, {'id': 'currency_conversion_5kxvpnv3h6fw', 'type': 'function', 'function': {'name': 'currency_conversion', 'arguments': '{"base_currency_value":10}'}}], 'token_count': {'input_tokens': 1559.0, 'output_tokens': 114.0}}, response_metadata={'id': '15b9ca31-f499-4bf7-8b25-19113ec151d2', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will first find the conversion factor between INR and USD. Then, I will use this factor to convert 10 USD to INR.'

In [192]:
messages.append(ai_message)

In [193]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I will first find the conversion factor between INR and USD. Then, I will use this factor to convert 10 USD to INR.', additional_kwargs={'id': '15b9ca31-f499-4bf7-8b25-19113ec151d2', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will first find the conversion factor between INR and USD. Then, I will use this factor to convert 10 USD to INR.', 'tool_calls': [{'id': 'get_conversion_factor_xzezjyvqevv2', 'type': 'function', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency":"INR","target_currency":"USD"}'}}, {'id': 'currency_conversion_5kxvpnv3h6fw', 'type': 'function', 'function': {'name': 'currency_conversion', 'arguments': '{"base_currency_value":10}'}}], 'token_count': {'input_tokens': 1559.0, 'output_tokens': 114.0}}, response_metadata={'id': '15b9ca31-f499-4bf7-8b25-19113

In [194]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'get_conversion_factor_xzezjyvqevv2',
  'type': 'tool_call'},
 {'name': 'currency_conversion',
  'args': {'base_currency_value': 10},
  'id': 'currency_conversion_5kxvpnv3h6fw',
  'type': 'tool_call'}]

In [195]:
for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'currency_conversion':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = currency_conversion.invoke(tool_call)
    messages.append(tool_message2)

In [196]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I will first find the conversion factor between INR and USD. Then, I will use this factor to convert 10 USD to INR.', additional_kwargs={'id': '15b9ca31-f499-4bf7-8b25-19113ec151d2', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'I will first find the conversion factor between INR and USD. Then, I will use this factor to convert 10 USD to INR.', 'tool_calls': [{'id': 'get_conversion_factor_xzezjyvqevv2', 'type': 'function', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency":"INR","target_currency":"USD"}'}}, {'id': 'currency_conversion_5kxvpnv3h6fw', 'type': 'function', 'function': {'name': 'currency_conversion', 'arguments': '{"base_currency_value":10}'}}], 'token_count': {'input_tokens': 1559.0, 'output_tokens': 114.0}}, response_metadata={'id': '15b9ca31-f499-4bf7-8b25-19113

In [197]:
model_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.01045.\n\nBased on this, 10 USD is equal to 0.1045 INR.'